<a href="https://colab.research.google.com/github/LorenzoBioinfo/BiomedicalAgenticAI/blob/main/BiomedicalAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Biomedical Agent

In questo notebook sviluppiamo un Agente in grado di rispondere a domande biomediche.

L'agente sarà in grado di fare ricerche sul web consultando i seguenti siti:

* arxiv
* PubMed
* UniProt

In [5]:
!pip install langchain openai python-dotenv arxiv requests beautifulsoup4
!pip3 install langchain-openai

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.1 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=fa1e637538364da25a085b2caf45920aee891e41de0d52782655aa515c072e4d
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19


In [3]:
import os
from dotenv import load_dotenv
import arxiv
import requests
from typing import List
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import getpass




In [13]:
from langchain import __version__ as langchain_version
print(f"LangChain version: {langchain_version}")

LangChain version: 1.2.12


In [4]:
from google.colab import userdata
os.environ['OPENAI_API_KEY']   = userdata.get('OPENAI_API_KEY')
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

### Language Model

In [5]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)

messages = [
    (
        "system",
        "You are a helpful assistant that supports scientific reasearch.",
    ),
    ("human", "Who discovered the DNA double helix structure?"),
]

ai_msg = llm.invoke(messages)

In [20]:
print(ai_msg.text)

The DNA double helix structure was discovered by James Watson and Francis Crick in 1953. Their discovery revolutionized the field of molecular biology and genetics.


### Tools

In [35]:
from langchain.messages import SystemMessage, HumanMessage
import arxiv
import requests
from langchain.tools import tool

@tool
def search_arxiv(query: str, max_results: int = 5) -> str:
    """
    Search arXiv for papers on a specific topic.
    Use this for biomedical ML research papers.
    Args:
        query: the search topic, e.g. 'CRISPR off-target effects'
        max_results: number of papers to return
    """
    try:
        client = arxiv.Client()
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance,
        )

        papers = []
        for paper in client.results(search):
            papers.append(
                f"Title: {paper.title}\n"
                f"Authors: {paper.authors[0].name} et al.\n"
                f"Published: {paper.published.year}\n"
                f"Summary: {paper.summary[:300]}...\n"
                f"URL: {paper.entry_id}\n"
            )

        return "\n---\n".join(papers) if papers else "No papers found."

    except Exception as e:
        return f"Error searching arXiv: {str(e)}"


@tool
def search_pubmed(query: str, max_results: int = 5) -> str:
    """
    Search PubMed for biomedical literature.
    Uses NCBI E-utilities API (free, no key required).
    Args:
        query: the search topic, e.g. 'CRISPR off-target effects mitigation'
        max_results: number of papers to return
    """
    try:

        search_resp = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={
                "db": "pubmed",
                "term": query,
                "retmax": max_results,
                "retmode": "json",
                "sort": "relevance",
            },
            timeout=10,
        )
        search_resp.raise_for_status()
        ids = search_resp.json()["esearchresult"]["idlist"]

        if not ids:
            return "No papers found on PubMed."


        fetch_resp = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={
                "db": "pubmed",
                "id": ",".join(ids),
                "retmode": "json",
            },
            timeout=10,
        )
        fetch_resp.raise_for_status()
        data = fetch_resp.json()["result"]

        results = []
        for uid in ids[:max_results]:
            article = data.get(uid, {})
            title = article.get("title", "N/A")
            authors = article.get("authors", [])
            first_author = authors[0].get("name", "N/A") if authors else "N/A"
            pub_date = article.get("pubdate", "N/A")
            source = article.get("source", "N/A")

            results.append(
                f"Title: {title}\n"
                f"Authors: {first_author} et al.\n"
                f"Journal: {source}\n"
                f"Published: {pub_date}\n"
                f"PMID: {uid}\n"
                f"URL: https://pubmed.ncbi.nlm.nih.gov/{uid}/\n"
            )

        return "\n---\n".join(results)

    except Exception as e:
        return f"Error searching PubMed: {str(e)}"

@tool
def fetch_protein_info(protein_name: str) -> str:
    """
    Get protein information from UniProt by exact gene name.
    Searches only human proteins (Homo sapiens).
    Args:
        protein_name: exact gene or protein name, e.g. 'TP53', 'BRCA1'
    """
    try:

        queries_to_try = [
            f"gene_exact:{protein_name} AND organism_id:9606 AND reviewed:true",
            f"gene_exact:{protein_name} AND organism_id:9606",
            f"gene:{protein_name} AND organism_id:9606 AND reviewed:true",
        ]

        for query in queries_to_try:
            response = requests.get(
                "https://rest.uniprot.org/uniprotkb/search",
                params={
                    "query": query,
                    "format": "json",
                    "size": 3,
                    "fields": "accession,id,gene_names,organism_name,protein_name,cc_function",
                },
                timeout=15,
            )

            if response.status_code != 200:
                continue

            results = response.json().get("results", [])
            if not results:
                continue

            best = None
            for p in results:
                genes = p.get("genes", [])
                gene_val = genes[0].get("geneName", {}).get("value", "") if genes else ""
                if gene_val.upper() == protein_name.upper():
                    best = p
                    break

            if best is None:
                continue


            protein_desc = best.get("proteinDescription", {})
            full_name = (
                protein_desc.get("recommendedName", {})
                .get("fullName", {})
                .get("value", "N/A")
            )
            genes = best.get("genes", [])
            gene_name = genes[0].get("geneName", {}).get("value", "N/A") if genes else "N/A"
            organism = best.get("organism", {}).get("scientificName", "N/A")
            accession = best.get("primaryAccession", "N/A")
            uniprot_id = best.get("uniProtkbId", "N/A")


            function_text = "N/A"
            for comment in best.get("comments", []):
                if comment.get("commentType") == "FUNCTION":
                    texts = comment.get("texts", [])
                    if texts:
                        function_text = texts[0].get("value", "N/A")[:500]
                    break

            return (
                f"UniProt ID: {uniprot_id}\n"
                f"Accession: {accession}\n"
                f"Protein name: {full_name}\n"
                f"Gene: {gene_name}\n"
                f"Organism: {organism}\n"
                f"Function: {function_text}\n"
                f"URL: https://www.uniprot.org/uniprotkb/{accession}\n"
            )

        return f"Protein '{protein_name}' not found in UniProt (human, reviewed)."

    except Exception as e:
        return f"Error fetching protein info: {str(e)}"

@tool
def analyze_biological_concept(concept: str) -> str:
    """
    Get LLM to explain biological concepts clearly.
    """
    prompt = f"""
    Explain this biomedical concept in clear, accurate terms:
    {concept}

    Include:
    1. Definition
    2. Biological significance
    3. Clinical relevance
    4. Examples
    """
    return llm.invoke([HumanMessage(content=prompt)])



@tool
def format_bibliography_tool(papers: str) -> str:
    """
    Get LLM to format scientific papers in APA style.
    """

    prompt = f"""
    Format these paper details as a proper scientific bibliography (APA style):

    {papers}
    """
    return llm.invoke([HumanMessage(content=prompt)])


In [36]:
from langchain.agents import create_agent
import langchain
langchain.debug=True
agent = create_agent(
        tools=[search_arxiv,search_pubmed,fetch_protein_info,analyze_biological_concept,format_bibliography_tool],
        model=llm,
        system_prompt="You are a helpful scientific assistant that support scientific research\
        You can use tools to search into Pubmed, arxiv e uniprot database. Use analyze biological concept to explain\
        in concise way")



In [37]:

queries = [
        "What are recent advances in CRISPR off-target effect mitigation? Search arXiv and PubMed.",
        "Tell me about the P53 protein and its role in cancer. Use the protein database.",
        "What is single-cell RNA-seq and why is it important? Explain the concept.",
    ]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*50}")
    print(f"Query {i}: {query}")
    print(f"{'='*50}\n")

    try:
        result = agent.invoke({"messages": [HumanMessage(query)]})
        ai_message = list(result['messages'])[-1]
        print(f"Result:\n {ai_message.content}")
    except Exception as e:
        print(f"Error: {str(e)}")

    print("\n")


Query 1: What are recent advances in CRISPR off-target effect mitigation? Search arXiv and PubMed.

Result:
 ### Recent Advances in CRISPR Off-Target Effect Mitigation

#### From arXiv:
1. **Title:** Comparative Analysis of Machine Learning Algorithms for Predicting On-Target and Off-Target Effects of CRISPR-Cas13d for gene editing
   - **Authors:** Jingze Liu et al.
   - **Published:** 2023
   - **Summary:** This study compared the performance of machine learning algorithms in predicting on-target and off-target effects of CRISPR-Cas13d. It aimed to assist in designing specific sgRNAs for locating desired RNA target positions.
   - [Link to Paper](http://arxiv.org/abs/2305.06769v1)

2. **Title:** Guide-Guard: Off-Target Predicting in CRISPR Applications
   - **Authors:** Joseph Bingham et al.
   - **Published:** 2026
   - **Summary:** Discusses cyber-physical genome sequencing and editing technologies like CRISPR, enabling researchers to investigate genetics and health science topics